In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, logging
import torch

tokenizer = AutoTokenizer.from_pretrained(r"hungnguyen190204/Test_LLM_6")
model = AutoModelForCausalLM.from_pretrained(r"hungnguyen190204/Test_LLM_6", low_cpu_mem_usage=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
logging.set_verbosity(logging.CRITICAL)
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)

2025-06-29 16:14:37.078262: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751213677.368637      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751213677.445042      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/554M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [3]:
import requests
import time

In [4]:
URL_NGROK = "https://c539-116-108-3-127.ngrok-free.app/".rstrip('/')
API_RECEIVE_QUESTION = f"{URL_NGROK}/get_kaggle_question"
API_SEND_ANSWER      = f"{URL_NGROK}/receive_answer"

In [5]:
print("⏳ Bắt đầu lắng nghe câu hỏi từ server...")

# while True:
    try:
        # Gọi API lấy câu hỏi (do Flask hoặc app bạn gửi lên trước đó)
        response = requests.get(API_RECEIVE_QUESTION, timeout=10)
        
        if response.status_code == 200 and response.json():
            data = response.json()
            prompt = data["message"]
            name_conv = data["name_conversation"]
            print(f"📩 Nhận câu hỏi: {prompt}")

            # Gọi model xử lý
            result = pipe(f"<s>[INST] {prompt} [/INST]")
            full_text = result[0]["generated_text"]
            answer = full_text.split('[/INST]')[-1].strip()

            print("🤖 Bot trả lời:", answer)

            # Gửi kết quả về máy bạn
            res = requests.post(API_SEND_ANSWER, json={
                "sender": "bot",
                "message": answer,
                "name_conversation": name_conv
            })

            if res.status_code == 200:
                print("✅ Gửi câu trả lời thành công về Flask")

        time.sleep(2)

    except Exception as e:
        print("⚠️ Lỗi:", e)
        time.sleep(5)

IndentationError: unexpected indent (983648104.py, line 4)